<a href="https://colab.research.google.com/github/whiiru/LAB-5-PROG-CONC/blob/main/C%C3%B3pia_de_Lab5_ProgramacaoConcorrente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Laboratório 5: Sincronização por Condição (Variáveis de Condição)**

**Disciplina:** Programação Concorrente (ICP-361) - UFRJ

## **Introdução**

O objetivo deste Laboratório é introduzir o mecanismo de sincronização por condição usando variáveis de condição da biblioteca Pthread.


In [ ]:
!gcc --version

gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



## **Preparação do Ambiente**

Como o Google Colab roda em Linux, podemos compilar e executar códigos C nativamente. Vamo verificar a versão do compilador gcc.


## **Atividade 1: Barreira**

**Objetivo:** Experimentar o padrão de sincronização coletiva (barreira).

O código abaixo simula o arquivo `barreira.c` mencionado no roteiro. Ele implementa uma barreira onde as threads devem esperar até que todas cheguem ao ponto de sincronização antes de prosseguir.

### **1. Criar o código da Barreira**

Execute a célula abaixo para criar o arquivo `barreira.c`.

In [ ]:
%%writefile barreira.c
#include <pthread.h>
#include <stdio.h>
#include <stdlib.h>
#include <unistd.h>

#define NTHREADS 5
#define PASSOS 4

pthread_barrier_t barreira_global;

void *tarefa(void *arg) {
    int id = *(int *)arg;
    int b;

    for (int i = 0; i < PASSOS; i++) {
        printf("Thread %d: passo %d\n", id, i);

        for (b = 0; b < 1000000; b++);

      //  int ret = pthread_barrier_wait(&barreira_global);
      //  if (ret != 0 && ret != PTHREAD_BARRIER_SERIAL_THREAD) {
      //      printf("Erro na barreira da thread %d\n", id);
      //      pthread_exit(NULL);
      //   }
    }

    pthread_exit(NULL);
}

int main(int argc, char *argv[]) {
    pthread_t threads[NTHREADS];
    int id[NTHREADS];

    if (pthread_barrier_init(&barreira_global, NULL, NTHREADS) != 0) {
        printf("Erro ao inicializar a barreira.\n");
        return 1;
    }

    for (int i = 0; i < NTHREADS; i++) {
        id[i] = i;
        pthread_create(&threads[i], NULL, tarefa, (void *)&id[i]);
    }

    for (int i = 0; i < NTHREADS; i++) {
        pthread_join(threads[i], NULL);
    }

    pthread_barrier_destroy(&barreira_global);
    printf("FIM.\n");
    return 0;
}

Overwriting barreira.c


### **2. Execução (Sem Barreira)**

Compile e execute o programa. Verifique se as threads estão sincronizadas ou desordenadas.

In [ ]:
!gcc -o barreira barreira.c -lpthread
!./barreira

### **3. Execução (Com Barreira)**

**Instrução:** Edite a célula do código `barreira.c` acima, **descomentando** as linhas comentadas. Depois, rode a célula de compilação abaixo novamente. Qual foi a diferença?

In [ ]:
# Recompile e execute após alterar o código
!gcc -o barreira barreira.c -lpthread
!./barreira

### **4. Execução (Com Barreira e número diferente de threads)**

**Instrução:** Edite a célula do código `barreira.c` acima, alterando a linha para:

```c
if (pthread_barrier_init(&barreira_global, NULL, NTHREADS+1) != 0)
```
e depois para:

```c
if (pthread_barrier_init(&barreira_global, NULL, NTHREADS-3) != 0)
```

  - Verifique o que acontece em cada caso.

In [ ]:
# Recompile e execute após alterar o código
!gcc -o barreira barreira.c -lpthread
!./barreira

## **Atividade 2: Hello Bye (Ordem de Execução)**

**Objetivo:** Usar variáveis de condição para controlar a ordem: imprimir "HELLO" antes de "BYEBYE".

### **1. Criar o código `hellobye.c`**

In [ ]:
%%writefile hellobye.c
/* Disciplina: Programação Concorrente */
/* Profa.: Silvana Rossetto */
/* Laboratório: 5 */
/* Codigo: Uso de variáveis de condição e suas operações básicas para sincronização por condição */
/* Condição lógica da aplicação: uma thread A deve imprimir HELLO antes da thread B imprimir BYEBYE */

#include <pthread.h>
#include <stdio.h>
#include <stdlib.h>

#define NTHREADS  2

/* Variaveis globais */
short int hello = 0;
pthread_mutex_t mutex;
pthread_cond_t cond;

/* Thread A */
void *A (void *t) {
  printf("A: Comecei\n");

  //faz uma computação qualquer...

  //executa ''hello'' (OU vai para o estado 'hello')
  printf("HELLO\n");

  //regitra transicao de estado (para o caso da outra thread ainda não ter chegado ao ponto que deve verificar o estado)
  pthread_mutex_lock(&mutex);
  hello=1; //registra a transição de estado
  //sinaliza outra thread (para o caso da outra thread já estar bloqueada)
  printf("A: vai sinalizar a condicao \n");
  pthread_cond_signal(&cond);
  pthread_mutex_unlock(&mutex);

  //faz outra computação qualquer...

  pthread_exit(NULL);
}

/* Thread B */
void *B (void *t) {
  printf("B: Comecei\n");

  //faz uma computação qualquer...

  //verifica se pode executar ''byebye''
  pthread_mutex_lock(&mutex);
  if (!hello) { // estado não transicionou ainda...
     printf("B: vai se bloquear para aguardar transicao de estado\n");
     pthread_cond_wait(&cond, &mutex); //bloqueia até estado valido
     printf("B: sinal recebido e mutex realocado \n");
  }
  pthread_mutex_unlock(&mutex);

  //executa ''byebye''
  printf("BYEBYE\n");

  //faz outra computação qualquer...

  pthread_exit(NULL);
}


/* Funcao principal */
int main(int argc, char *argv[]) {
  pthread_t threads[NTHREADS];

  /* Inicializa o mutex (lock de exclusao mutua) e a variavel de condicao */
  pthread_mutex_init(&mutex, NULL);
  pthread_cond_init (&cond, NULL);

  /* Cria as threads */
    pthread_create(&threads[1], NULL, B, NULL);
    pthread_create(&threads[0], NULL, A, NULL);

  /* Espera todas as threads completarem */
  for (int i = 0; i < NTHREADS; i++) {
    pthread_join(threads[i], NULL);
  }
  printf ("\nFIM\n");

  /* Desaloca variaveis e termina */
  pthread_mutex_destroy(&mutex);
  pthread_cond_destroy(&cond);
  return 0;
}

Writing hellobye.c



### **2. Execução e Testes**

1. Abra o arquivo hello bye.c e identifique qual é o requisito lógico/condicional da aplicação  (qual é a ordem de impressão requerida para as expressões HELLO e BYE). Acompanhe a explanação.
2. Execute a aplicação várias vezes e verifique se o requisito lógico é sempre cumprido.
3. Inverta a ordem de criação das threads A e B e execute o programa novamente. Verifique os resultados.

In [ ]:
!gcc -o hellobye hellobye.c -lpthread
!./hellobye

## **Atividade 3: Hello 2 Bye**

**Objetivo:** Duas threads imprimem "HELLO", uma thread espera ambas para imprimir "BYEBYE".

### **1. Criar código `hello2bye.c`**

In [ ]:
%%writefile hello2bye.c
#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>

/* Variaveis globais */
int x = 0;
pthread_mutex_t x_mutex;
pthread_cond_t x_cond;

/* Thread A (Imprime Hello) */
void *A (void *t) {
  printf("HELLO\n");

  pthread_mutex_lock(&x_mutex);
  x++; //incrementa contador de hellos
  if(x==2) { //se for o segundo hello, acorda quem espera
      pthread_cond_signal(&x_cond);
  }
  pthread_mutex_unlock(&x_mutex);

  pthread_exit(NULL);
}

/* Thread B (Espera 2 Hellos para imprimir Bye) */
void *B (void *t) {
  pthread_mutex_lock(&x_mutex);
  while (x < 2) {
     pthread_cond_wait(&x_cond, &x_mutex);
  }
  pthread_mutex_unlock(&x_mutex);

  printf("BYEBYE\n");

  pthread_exit(NULL);
}

int main(int argc, char *argv[]) {
  pthread_t threads[3];

  pthread_mutex_init(&x_mutex, NULL);
  pthread_cond_init(&x_cond, NULL);

  /* Criar 2 threads A e 1 thread B */
  pthread_create(&threads[0], NULL, A, NULL);
  pthread_create(&threads[1], NULL, A, NULL);
  pthread_create(&threads[2], NULL, B, NULL);

  for (int i = 0; i < 3; i++) {
    pthread_join(threads[i], NULL);
  }

  pthread_mutex_destroy(&x_mutex);
  pthread_cond_destroy(&x_cond);
  return 0;
}

Overwriting hello2bye.c


### **2. Execução**

Verifique se o "BYEBYE" só aparece após dois "HELLOs".

In [ ]:
!gcc -o hello2bye hello2bye.c -lpthread
!./hello2bye

HELLO
HELLO
BYEBYE
^C


## **Atividade 4: Many to Many**

**Objetivo:** Duas threads A e duas threads B. As duas B devem esperar as duas A executarem "HELLO".

### **1. Implementação (Exercício)**

Complete o código da atividade acima para atender ao requisito.
**Dica:** O contador `x` deve chegar a 2, mas agora temos duas threads esperando. O que fazer?

### **2. Execução**

In [ ]:
!gcc -o atividade4 atividade4.c -lpthread
!./atividade4

## **Atividade 5: Método de Jacobi

**Objetivo:**

### **Espaço para Implementação**

Utilize o bloco abaixo para desenvolver a solução que será enviada.

In [ ]:
%%writefile jacobi.c
#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>
#include <math.h>

#define N 10              // Tamanho da matriz aumentado
#define MAX_IT 1000
#define TOL 1e-6
#define NTHREADS 2

// Matriz A (Diagonal Dominante para garantir convergência)
double A[N][N] = {
    {20, -1, -1,  0,  0,  0,  0,  0,  0,  0},
    {-1, 15, -2, -1,  0,  0,  0,  0,  0,  0},
    {-1, -2, 18, -1, -1,  0,  0,  0,  0,  0},
    { 0, -1, -1, 12, -1, -1,  0,  0,  0,  0},
    { 0,  0, -1, -1, 14, -1, -1,  0,  0,  0},
    { 0,  0,  0, -1, -1, 16, -1, -1,  0,  0},
    { 0,  0,  0,  0, -1, -1, 13, -1, -1,  0},
    { 0,  0,  0,  0,  0, -1, -1, 17, -2, -1},
    { 0,  0,  0,  0,  0,  0, -1, -2, 15, -1},
    { 0,  0,  0,  0,  0,  0,  0, -1, -1, 10}
};

double b[N] = {18, 11, 13, 8, 10, 12, 9, 12, 11, 8};

double x[N];
double x_new[N];
int global_converged = 0; // Flag para encerrar todas as threads

pthread_barrier_t barrier;

typedef struct {
    int id;
    int start;
    int end;
} thread_data_t;

void* jacobi_thread(void* arg) {
    thread_data_t* data = (thread_data_t*) arg;
    int i, j, k;

    for (k = 0; k < MAX_IT; k++) {
        // Etapa 1: Calcular x_new baseado no x da iteração anterior
        for (i = data->start; i < data->end; i++) {
            double sigma = 0.0;
            for (j = 0; j < N; j++) {
                if (j != i) {
                    sigma += A[i][j] * x[j];
                }
            }
            x_new[i] = (b[i] - sigma) / A[i][i];
        }

        // Barreira 1: Esperar todos calcularem x_new antes de verificar erro/atualizar
        pthread_barrier_wait(&barrier);

        // Etapa 2: Critério de parada (antes de sobrescrever x)
        if (data->id == 0) {
            double erro = 0.0;
            for (i = 0; i < N; i++) {
                erro += fabs(x_new[i] - x[i]);
            }
            if (erro < TOL) {
                printf("Convergiu em %d iteracoes. Erro: %e\n", k + 1, erro);
                global_converged = 1;
            }
        }

        // Barreira 2: Garantir que a thread 0 terminou a verificação
        pthread_barrier_wait(&barrier);
        if (global_converged) break;

        // Etapa 3: Atualizar x para a próxima iteração
        for (i = data->start; i < data->end; i++) {
            x[i] = x_new[i];
        }

        // Barreira 3: Garantir que x foi totalmente atualizado antes da próxima iteração
        pthread_barrier_wait(&barrier);
    }

    pthread_exit(NULL);
}

int main() {
    pthread_t threads[NTHREADS];
    thread_data_t data[NTHREADS];

    for (int i = 0; i < N; i++) x[i] = 0.0;

    pthread_barrier_init(&barrier, NULL, NTHREADS);

    int chunk = N / NTHREADS;

    for (int t = 0; t < NTHREADS; t++) {
        data[t].id = t;
        data[t].start = t * chunk;
        data[t].end = (t == NTHREADS - 1) ? N : (t + 1) * chunk;
        pthread_create(&threads[t], NULL, jacobi_thread, &data[t]);
    }

    for (int t = 0; t < NTHREADS; t++) {
        pthread_join(threads[t], NULL);
    }

    pthread_barrier_destroy(&barrier);

    printf("\nSolucao aproximada:\n");
    for (int i = 0; i < N; i++) {
        printf("x[%d] = %f\n", i, x[i]);
    }

    return 0;
}

Overwriting jacobi.c


In [ ]:
!gcc -o teste jacobi.c -lpthread
!./teste

Convergiu em 14 iteracoes. Erro: 3.088458e-07

Solucao aproximada:
x[0] = 1.000000
x[1] = 1.000000
x[2] = 1.000000
x[3] = 1.000000
x[4] = 1.000000
x[5] = 1.000000
x[6] = 1.000000
x[7] = 1.000000
x[8] = 1.000000
x[9] = 1.000000


## **Atividade 6: Aplicação em Soma (Entrega)**

**Objetivo:** Alterar o programa `soma-lock-atom.c` (do Lab 4). A thread `ExecutaTarefa` soma valores. A cada múltiplo de 1000, ela deve pausar e esperar uma thread `Extra` imprimir o valor atual.

### **Requisito do Código:**

1. Thread `ExecutaTarefa`: Soma números. Se `soma % 1000 == 0`, sinaliza a thread Extra e dorme.
2. Thread `Extra`: Dorme até receber sinal. Imprime a soma. Sinaliza `ExecutaTarefa` para continuar.

### **Espaço para Implementação**

Utilize o bloco abaixo para desenvolver a solução que será enviada.

In [ ]:
%%writefile lab6_entrega.c
/* Disciplina: Programacao Concorrente */
/* Prof.: Silvana Rossetto */
/* Codigo: Comunicação entre threads usando variável compartilhada e exclusao mutua com bloqueio */

#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>

long int soma = 0; //variavel compartilhada entre as threads
pthread_mutex_t mutex; //variavel de lock para exclusao mutua

//funcao executada pelas threads
void *ExecutaTarefa (void *arg) {
  int id = *(int *) arg;
  printf("Thread : %d esta executando...\n", id);

  for (int i=0; i<100000; i++) {
     //--entrada na SC
     pthread_mutex_lock(&mutex);
     //--SC (seção critica)
     soma++; //incrementa a variavel compartilhada
     //--saida da SC
     pthread_mutex_unlock(&mutex);
  }
  printf("Thread : %d terminou!\n", id);
  pthread_exit(NULL);
}

//funcao executada pela thread de log
void *extra (void *args) {
  printf("Extra : esta executando...\n");
  for (int i=0; i<10000; i++) {
     if (!(soma%10)) //imprime se 'soma' for multiplo de 10
        printf("soma = %ld \n", soma);
  }
  printf("Extra : terminou!\n");
  pthread_exit(NULL);
}

//fluxo principal
int main(int argc, char *argv[]) {
   pthread_t *tid; //identificadores das threads no sistema
   int nthreads; //qtde de threads (passada linha de comando)

   //--le e avalia os parametros de entrada
   if(argc<2) {
      printf("Digite: %s <numero de threads>\n", argv[0]);
      return 1;
   }
   nthreads = atoi(argv[1]);
   int id[nthreads];

   //--aloca as estruturas
   tid = (pthread_t*) malloc(sizeof(pthread_t)*(nthreads+1));
   if(tid==NULL) {puts("ERRO--malloc"); return 2;}

   //--inicilaiza o mutex (lock de exclusao mutua)
   pthread_mutex_init(&mutex, NULL);

   //--cria as threads
   for(int t=0; t<nthreads; t++) {
     id[t] = t;
     if (pthread_create(&tid[t], NULL, ExecutaTarefa, &id[t])) {
       printf("--ERRO: pthread_create()\n"); exit(-1);
     }
   }

   //--cria thread de log
   if (pthread_create(&tid[nthreads], NULL, extra, NULL)) {
      printf("--ERRO: pthread_create()\n"); exit(-1);
   }

   //--espera todas as threads terminarem
   for (int t=0; t<nthreads+1; t++) {
     if (pthread_join(tid[t], NULL)) {
         printf("--ERRO: pthread_join() \n"); exit(-1);
     }
   }

   //--finaliza o mutex
   pthread_mutex_destroy(&mutex);

   printf("Valor de 'soma' = %ld\n", soma);

   return 0;
}

Overwriting lab6_entrega.c



### **Testar Solução**

In [ ]:
!gcc -o lab6_entrega lab6_entrega.c -lpthread
!./lab6_entrega 5


Thread : 3 esta executando...
Extra : esta executando...
soma = 24091 
soma = 24321 
soma = 24491 
soma = 24671 
soma = 24830 
soma = 25000 
soma = 25171 
soma = 25340 
soma = 25511 
soma = 25681 
soma = 25851 
soma = 26020 
soma = 26190 
soma = 26360 
soma = 26530 
soma = 26701 
soma = 26870 
soma = 27040 
soma = 27210 
soma = 27381 
soma = 27550 
soma = 27720 
soma = 27891 
soma = 28061 
soma = 28221 
soma = 28390 
soma = 28560 
soma = 28730 
soma = 28900 
soma = 29070 
soma = 29240 
soma = 29410 
soma = 29580 
soma = 29771 
soma = 29930 
soma = 30100 
soma = 30270 
soma = 30441 
soma = 30610 
soma = 30790 
soma = 30961 
soma = 31130 
soma = 31300 
soma = 31471 
soma = 31641 
soma = 31810 
soma = 31981 
soma = 32150 
soma = 32320 
soma = 32490 
soma = 32660 
soma = 32830 
soma = 33000 
soma = 33171 
soma = 33340 
soma = 33510 
soma = 33680 
soma = 33850 
soma = 34020 
soma = 34190 
soma = 34360 
soma = 34530 
soma = 34700 
soma = 35281 
soma = 35451 
soma = 35621 
soma = 35790 
soma 

## **Entrega**

Disponibilize o código implementado na **Atividade 5** em um ambiente de acesso remoto (GitHub ou GitLab). Use o formulário de entrega para enviar o link.
